# MoLoRAG Reproduction: Baseline 2 (Multi-modal M3DocRAG)

This is **Version 1.6.5** - No Login Required.
**Fixes**: Solves the `GatedRepoError` (401 Unauthorized) by loading the base model directly from the vidore repository. No HuggingFace login is needed anymore, but the 4-bit memory fixes are still active.

In [ ]:
# 1. Setup Environment
!apt-get update && apt-get install -y poppler-utils
!git clone https://github.com/WxxShirley/MoLoRAG.git
%cd MoLoRAG
!pip install -U transformers peft accelerate bitsandbytes
!pip install "requests==2.32.4" faiss-cpu langchain langchain-huggingface langchain-community pypdf PyMuPDF sentence-transformers huggingface_hub colpali_engine qwen_vl_utils pdf2image

In [ ]:
# 2. Download Dataset
import os
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs('dataset/MMLong', exist_ok=True)
print('Downloading samples_MMLong.json...')
try:
    hf_hub_download(repo_id='xxwu/MoLoRAG', repo_type='dataset', filename='samples_MMLong.json', local_dir='./dataset/')
except Exception as e:
    print(f'HF Download Error: {e}')

print('Downloading PDF subset...')
try:
    snapshot_download(repo_id='xxwu/MoLoRAG', repo_type='dataset', 
                      allow_patterns=['MMLong/*.pdf'], 
                      local_dir='./dataset/')
except Exception as e:
    print(f'HF Download Error: {e}')

In [ ]:
# 3. Create the Local Scripts (with Non-Gated Fix)
import os
os.makedirs('VLMModels', exist_ok=True)
os.makedirs('VLMRetriever', exist_ok=True)
os.makedirs('tmp/tmp_embs/MMLong', exist_ok=True)
os.makedirs('tmp/tmp_imgs/MMLong', exist_ok=True)

In [ ]:
print('Writing VLMModels/Qwen_VL_local.py...')
dir_name = 'VLMModels'
if dir_name: os.makedirs(dir_name, exist_ok=True)
code = 'from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor\nfrom qwen_vl_utils import process_vision_info\nimport torch\nimport os\n\ndef format_image_path(raw_path):\n    # Ensure current working directory is included\n    abs_path = os.path.abspath(raw_path)\n    return f"file://{abs_path}" \n\ndef init_model(model_name, device=torch.device("cuda")):\n    if "lora" in model_name: \n        model_path = "xxwu/MoLoRAG-QwenVL-3B"\n        print(f"Loading LoRA model from {model_path}")\n    elif "3B" in model_name:\n        model_path =  "Qwen/Qwen2.5-VL-3B-Instruct"\n    elif "7B" in model_name:\n        model_path = "Qwen/Qwen2.5-VL-7B-Instruct"\n\n    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n        model_path,\n        torch_dtype=torch.float16,\n        # Use flash_attention_2 if available, otherwise default\n        # T4 supports it if installed, but fallback is safer\n        device_map="auto"\n    ).eval()\n    \n    min_pixels = 256 * 28 * 28 \n    max_pixels = 512 * 28 * 28\n    processor = AutoProcessor.from_pretrained(model_path, min_pixels=min_pixels, max_pixels=max_pixels)\n    model.processor = processor\n    \n    return model \n\ndef get_response_concat(model, question, image_path_list, max_new_tokens=1024, temperature=1.0):\n    msgs = []\n\n    if isinstance(image_path_list, list):\n        msgs.extend([dict(type=\'image\', image=format_image_path(p)) for p in image_path_list])\n    else:\n        msgs = [dict(type=\'image\', image=format_image_path(image_path_list))]\n    \n    msgs.append(dict(type=\'text\', text=question))\n    messages = [{ "role": "user", "content": msgs }]\n\n    text = model.processor.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True\n    )\n    image_inputs, video_inputs = process_vision_info(messages)\n\n    inputs = model.processor(\n        text=[text],\n        images=image_inputs,\n        video_inputs=video_inputs,\n        padding=True, \n        return_tensors=\'pt\'\n    )\n    inputs = inputs.to(model.device)\n    \n    with torch.no_grad():\n        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)\n    \n    generated_ids_trimmed = [\n        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)\n    ]\n    \n    output_text = model.processor.batch_decode(\n        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False\n    )\n\n    return output_text[0]\n'
with open('VLMModels/Qwen_VL_local.py', 'w') as f:
    f.write(code)

In [ ]:
print('Writing VLMRetriever/index_local.py...')
dir_name = 'VLMRetriever'
if dir_name: os.makedirs(dir_name, exist_ok=True)
code = 'from colpali_engine.models import ColPali, ColPaliProcessor\nfrom transformers import PaliGemmaForConditionalGeneration, BitsAndBytesConfig\nimport os \nfrom pdf2image import convert_from_path\nimport argparse\nimport torch\nfrom tqdm import tqdm\nimport sys \nsys.path.append("../")\nfrom utils import prepare_files\n\ndef encode_document(doc_path, doc_id, batch_size=1, resolution=100, save_emb=True, save_img=True):\n    # Reduced resolution to 100 DPI for safe T4 execution\n    try:\n        page_images = convert_from_path(doc_path, dpi=resolution)\n    except Exception as e:\n        print(f"Error converting {doc_path}: {e}")\n        return\n    \n    if save_img: \n        os.makedirs(img_save_dir, exist_ok=True)\n        for page_num, page_snapshot in enumerate(page_images):\n            img_path = f"{img_save_dir}/{doc_id}-{page_num+1}.png"\n            if not os.path.exists(img_path):\n                page_snapshot.save(img_path) \n\n    if save_emb:\n        total_image_embeds = torch.Tensor().to(device)\n        for idx in range(0, len(page_images), batch_size):\n            batch_images = page_images[idx: idx+batch_size]\n            batch_images = processor.process_images(batch_images).to(device)\n            with torch.no_grad():\n                image_embeds = model(**batch_images)\n            # Ensure we only store the embeddings on CPU to save VRAM\n            total_image_embeds = torch.cat((total_image_embeds, image_embeds.to(device)), dim=0)\n            \n            torch.cuda.empty_cache()\n            import gc; gc.collect()\n        \n        torch.save(total_image_embeds.cpu(), f"{save_dir}/{doc_id}.pt")\n        print(f"Saved Embeddings {total_image_embeds.shape} to {save_dir}/{doc_id}.pt")\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dataset", type=str, default="MMLong")\n    parser.add_argument("--save_dir", type=str, default="../tmp/tmp_embs")\n    parser.add_argument("--img_save_dir", type=str, default="../tmp/tmp_imgs")\n    parser.add_argument("--batch_size", type=int, default=1)\n    parser.add_argument("--model_name", type=str, default="vidore/colpali-v1.2")\n    args = parser.parse_args()\n    \n    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"\n    \n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Loading ColPali via PaliGemma base on {device}...")\n    \n    # Switch to float16 to avoid bitsandbytes AssertionError on T4 GPUs\n    # Use device_map=\'auto\' to let accelerate handle placement.\n    # Manual .to(device) is REMOVED to avoid NotImplementedError (Meta Tensor).\n    model = ColPali.from_pretrained(\n        "vidore/colpaligemma-3b-pt-448-base",\n        torch_dtype=torch.float16,\n        device_map="auto"\n    )\n\n    # Load the adapter into the ColPali instance\n    print(f"Loading adapter: {args.model_name}")\n    model.load_adapter(args.model_name)\n    \n    # model.to(device) removed to avoid Meta Tensor failure.\n    # device_map=\'auto\' already handled placement.\n    model.eval()\n    \n    processor = ColPaliProcessor.from_pretrained(args.model_name)\n    \n\n    documents = prepare_files(f"../dataset/{args.dataset}", suffix=".pdf")\n    print(f"Found {len(documents)} PDF(s) in ../dataset/{args.dataset}")\n    \n    documents = documents[:5] # Subset logic\n    print(f"Encoding subset: {documents}")\n    \n    save_dir, img_save_dir = f"{args.save_dir}/{args.dataset}", f"{args.img_save_dir}/{args.dataset}"\n    os.makedirs(save_dir, exist_ok=True)\n\n    for doc_path in tqdm(documents, desc="Multi-modal Encoding (Subset)"):\n        doc_id = doc_path.replace(".pdf", "") \n        print(f"Processing {doc_id}...")\n        encode_document(doc_path=f"../dataset/{args.dataset}/{doc_path}", doc_id=doc_id, save_img=True)\n'
with open('VLMRetriever/index_local.py', 'w') as f:
    f.write(code)

In [ ]:
print('Writing VLMRetriever/retrieve_local.py...')
dir_name = 'VLMRetriever'
if dir_name: os.makedirs(dir_name, exist_ok=True)
code = 'import torch \nfrom tqdm import tqdm \nfrom colpali_engine.models import ColPali, ColPaliProcessor\nimport argparse\nimport sys \nsys.path.append("../")\nfrom utils import load_all_doc_embeddings\nimport json \nimport os \n\nclass DocumentRetriever:\n    def __init__(self, encoder, processor, device, batch_size=128):\n        self.encoder = encoder \n        self.processor = processor \n        self.device = device \n        self.batch_size = batch_size     \n\n    def compute_scores(self, query, all_embeds):\n        queries = self.processor.process_queries(queries=[query]).to(self.device)\n        with torch.no_grad():\n            query_embeds = self.encoder(**queries)\n        \n        all_embeds = all_embeds.to(device=self.device, dtype=query_embeds.dtype)\n        with torch.no_grad():\n            scores = self.processor.score_multi_vector(query_embeds, all_embeds)\n            if len(scores.shape) > 1:\n                scores = scores[0]\n        return scores.cpu()\n    \n    def base_retrieve(self, query, all_embeds, top_k=5):\n        scores = self.compute_scores(query, all_embeds)\n        top_indices = scores.argsort(dim=-1, descending=True)[:top_k].tolist()\n        top_scores = scores[top_indices].tolist()\n        return [idx+1 for idx in top_indices], top_scores\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dataset", type=str, default="MMLong")\n    parser.add_argument("--emb_root", type=str, default="../tmp/tmp_embs")\n    parser.add_argument("--top_k", type=int, default=5)\n    args = parser.parse_args()\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    doc2emb = load_all_doc_embeddings(f"{args.emb_root}/{args.dataset}")\n    \n    model_name = "vidore/colpali-v1.2"\n    print(f"Loading ColPali via PaliGemma base on {device}...")\n    \n    print(f"Loading ColPali via PaliGemma base on {device}...")\n    \n    # Switch to float16 to avoid bitsandbytes AssertionError on T4 GPUs\n    # Removed device_map to avoid meta-tensor NotImplementedError during .to(device)\n    model = ColPali.from_pretrained(\n        "vidore/colpaligemma-3b-pt-448-base",\n        torch_dtype=torch.float16,\n        device_map="auto"\n    )\n\n    # Load the adapter into the ColPali instance\n    print(f"Loading adapter: {model_name}")\n    model.load_adapter(model_name)\n    \n    # model.to(device) removed to avoid Meta Tensor failure.\n    # device_map=\'auto\' already handled placement.\n    model.eval()\n    \n    processor = ColPaliProcessor.from_pretrained(model_name)\n    doc_retriever = DocumentRetriever(encoder=model, processor=processor, device=device)\n    \n    # Intensive cache clearing\n    torch.cuda.empty_cache()\n    import gc; gc.collect()\n    \n    samples_file = f"../dataset/samples_{args.dataset}.json"\n    samples = json.load(open(samples_file, \'r\'))\n    print(f"Loaded {len(samples)} samples from {samples_file}")\n    \n    # Filter for first 5 docs\n    emb_files = os.listdir(f"{args.emb_root}/{args.dataset}")\n    subset_docs = [f.replace(\'.pt\', \'\') for f in emb_files if f.endswith(\'.pt\')]\n    print(f"Found {len(subset_docs)} embedding file(s) in {args.emb_root}/{args.dataset}: {subset_docs}")\n    \n    filtered_samples = []\n    for s in samples:\n        clean_id = s[\'doc_id\'].replace(\'.pdf\', \'\')\n        if clean_id in subset_docs:\n            filtered_samples.append(s)\n    \n    samples = filtered_samples\n    print(f"Samples remaining after filtering for subset: {len(samples)}")\n    if len(samples) == 0:\n        print("Warning: 0 samples matched the subset_docs ID. Checking first sample ID format...")\n        if len(samples) > 0:\n            print(f"Example target ID: {samples[0][\'doc_id\'].replace(\'.pdf\', \'\')}")\n\n    retrieve_file = f"../dataset/retrieved/samples_{args.dataset}_base_local.json"\n    os.makedirs(os.path.dirname(retrieve_file), exist_ok=True)\n\n    for sample in tqdm(samples, desc="Multi-modal Retrieval"):\n        query = sample["question"]\n        doc_id = sample["doc_id"].replace(".pdf", "")\n        ranked_pages, page_scores = doc_retriever.base_retrieve(query, doc2emb[doc_id], top_k=args.top_k)\n        \n        sample["pages_ranking"] = str(ranked_pages)\n        sample["pages_scores"] = str(page_scores)\n\n    json.dump(samples, open(retrieve_file, \'w\'), indent=4)\n    print(f"Results saved to {retrieve_file}")\n'
with open('VLMRetriever/retrieve_local.py', 'w') as f:
    f.write(code)

In [ ]:
print('Writing main_vlm_local.py...')
dir_name = ''
if dir_name: os.makedirs(dir_name, exist_ok=True)
code = 'import os\nimport json\nimport argparse\nimport torch \nfrom tqdm import tqdm\nimport time \n\ndef main_vlm_local_QA(args):\n    st_time = time.time()\n    \n    from VLMModels.Qwen_VL_local import init_model, get_response_concat\n    \n    input_path = f"./dataset/retrieved/samples_{args.dataset}_base_local.json"\n    if not os.path.exists(input_path):\n        print(f"Error: {input_path} not found. Run retrieval first.")\n        return\n\n    samples = json.load(open(input_path, "r"))\n    \n    print(f"Loading local VLM: {args.model_name}")\n    model = init_model(args.model_name, device="auto")\n\n    for sample in tqdm(samples, desc="VLM QA"):\n        doc_id = sample[\'doc_id\'].replace(\'.pdf\', \'\')\n        ranked_pages = eval(sample["pages_ranking"])[:args.topk]\n        \n        # Paths to page images\n        input_image_list = [f"./tmp/tmp_imgs/{args.dataset}/{doc_id}-{p}.png" for p in ranked_pages]\n        \n        try:\n            query_prompt = f"Based on the images from the document, please answer the question: {sample[\'question\']}"\n            response = get_response_concat(model, query_prompt, input_image_list, max_new_tokens=128)\n        except Exception as e:\n            print(f"[ERROR] VLM prediction: {e}")\n            response = "None"\n\n        sample[args.response_key] = response \n     \n    output_path = f"./results/{args.dataset}/VLM/qwen2.5_vl_base_local.json"\n    os.makedirs(os.path.dirname(output_path), exist_ok=True)\n    with open(output_path, \'w\') as file:\n        json.dump(samples, file, indent=4)\n        \n    print(f"Completed in {(time.time() - st_time)/60:.2f} Mins")\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dataset", type=str, default="MMLong")\n    parser.add_argument("--model_name", type=str, default="QwenVL-3B")\n    parser.add_argument("--topk", type=int, default=3)\n    parser.add_argument("--response_key", type=str, default="raw_response")\n    args = parser.parse_args()\n    main_vlm_local_QA(args)\n'
with open('main_vlm_local.py', 'w') as f:
    f.write(code)

In [ ]:
# 4. Run Multi-modal Indexing
%cd VLMRetriever
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
!python3 index_local.py --dataset MMLong

In [ ]:
# 5. Run Multi-modal Retrieval
!python3 retrieve_local.py --dataset MMLong
import torch; torch.cuda.empty_cache()

In [ ]:
# 6. Run VLM QA
%cd ..
!python3 main_vlm_local.py --dataset MMLong --topk 3

In [ ]:
# 7. Results Overview
import json
import os
output_path = 'results/MMLong/VLM/qwen2.5_vl_base_local.json'
if os.path.exists(output_path):
    with open(output_path, 'r') as f:
        results = json.load(f)
        print(f'Total results: {len(results)}')
        print(json.dumps(results[:2], indent=2))
else:
    print('Output file not found.')

## 8. Comparison with MoLoRAG Paper (M3DocRAG Benchmarks)

The following tables show the **M3DocRAG** and **MoLoRAG** benchmarks from the original paper (EMNLP 2025).

In [ ]:
import pandas as pd

print("--- PAPER BENCHMARKS: QA Accuracy (%) ---")
paper_acc = {
    'Method': ['M3DocRAG (Paper)', 'MoLoRAG (Paper)', 'MoLoRAG+ (Paper)'],
    'MMLongBench': [29.11, 32.11, 32.47],
    'LongDocURL': [44.40, 45.79, 45.27]
}
display(pd.DataFrame(paper_acc))

print("\n--- PAPER BENCHMARKS: Retrieval Performance (Top-3) ---")
paper_retrieval = {
    'Metric': ['Recall (%)', 'Precision (%)', 'NDCG (%)', 'MRR (%)'],
    'M3DocRAG (MMLong)': [64.17, 31.62, 54.13, 65.36], 
    'MoLoRAG (MMLong)': [67.22, 40.81, 57.34, 68.56],
    'M3DocRAG (LongDocURL)': [67.00, 33.78, 58.23, 72.51],
    'MoLoRAG (LongDocURL)': [70.04, 36.41, 61.56, 75.78]
}
display(pd.DataFrame(paper_retrieval))

print("\n--- YOUR LOCAL REPRODUCTION (Subset) ---")
print("Check your metrics from the Final Report notebook to compare.")